In [1]:
from agents import Agent, Production, Chat, Toolkit, Prompt, Production
from pydantic import BaseModel

### **Toolkit**

Toolkits are MCP servers that are externally run. Before initializing a Toolkit object, the MCP server needs to be operational.

In [2]:
utils_toolkit = Toolkit(name = 'Utilities', url = 'http://localhost:9001/mcp')
iris_toolkit = Toolkit(name='IRIS', url = 'http://localhost:9002/mcp')


Load started on 04/08/2026 14:49:07
Loading file Agents.Message.ToolRequest.cls as udl
Compiling class Agents.Message.ToolRequest
Compiling table Agents_Message.ToolRequest
Compiling routine Agents.Message.ToolRequest.1
Load finished successfully.

Load started on 04/08/2026 14:49:07
Loading file Agents.Message.ToolResponse.cls as udl
Compiling class Agents.Message.ToolResponse
Compiling table Agents_Message.ToolResponse
Compiling routine Agents.Message.ToolResponse.1
Load finished successfully.

Load started on 04/08/2026 14:49:08
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/08/2026 14:49:08
Loading file Agents.Operation.ToolkitUtilities.cls as udl
Compiling class Agents.Operation.ToolkitUtilities
Compiling routine Agents.Operation.ToolkitUtilities.1
Load finished successfully.

Load started on 04/08/2026 14:49:08
Loading file Agents.Message.ToolRequest.cls as ud

### **Chat**

The Chat API can be used to persist conversations. A chat id can be used to construct a history of that Chat from IRIS instead of needing to maintain it manually. This is particularly important when Enterprise licenses for OpenAI have Zero Data Retention enabled and so OpenAI is not authorized to store the conversation on their servers, the Chat API allows for constructing the conversation from history stored in IRIS.

In [3]:
context = Chat(
    name="travel",
    messages=[
        {"role": "system", "content": "You are helpful."},
        {"role": "user", "content": "We are in Washington DC"},
        {"role": "assistant", "content": "Great, what do you want to do in DC?"}
    ]
)
context

Chat(name='travel', messages=3)

In [4]:
context.messages

[{'role': 'system', 'content': 'You are helpful.'},
 {'role': 'user', 'content': 'We are in Washington DC'},
 {'role': 'assistant', 'content': 'Great, what do you want to do in DC?'}]

In [5]:
context == Chat('travel')

True

### **Prompt**

- The Prompt API is a way to manage and version Prompts. 
- Prompts can be built at runtime using parameters. 
- Prompts Prompts versions can be fetched by a selected version. 
- Variables contained in a prompt can be queried using `get_variables()` method.

In [6]:
bond_system = Prompt(name = 'Agent007', text = 'You are {agent_name}. You always stay in character.')
bond_system.build(agent_name='James Bond')

'You are James Bond. You always stay in character.'

In [7]:
bond_system = Prompt(name = 'Agent007', text = 'Your next mission is of utmost importance, you do not have time to talk.')
bond_system

Prompt(name='Agent007', version=2, text='Your next mission is of utmost importance, you do not have time to talk.')

In [8]:
Prompt('Agent007') == bond_system

True

In [9]:
Prompt('Agent007', version=1)

Prompt(name='Agent007', version=1, text='You are {agent_name}. You always stay in character.')

In [10]:
Prompt('Agent007', version=1).get_variables()

['agent_name']

In [11]:
Prompt('Agent007').delete()
try:
    prompt = Prompt("Agent007")
except ValueError as e:
    print(e)

No prompt found for 'Agent007'


### **Agents**

Agents can be defined by a name, a description (not currently used in any way but can be leveraged in the future for expert selection), and an OpenAI model. Optionally, agents can be configured with a default structured output (modifiable at call time) and a set of toolkits the agent should have access to. These tools are advertised to the LLM specific to access the agent has at a Toolkit level (specifying individual tools inside a Toolkit is not currently supported). Agents must be added to a Production before being used.

In [12]:
molly = Agent(name='Molly', model='gpt-5')
Production('AgentSpace', [molly]).start()
molly('What are some summer hiking trails around Boston?')


Load started on 04/08/2026 14:49:10
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/08/2026 14:49:10
Loading file Agents.Utils.Production.cls as udl
Compiling class Agents.Utils.Production
Compiling routine Agents.Utils.Production.1
Load finished successfully.

Load started on 04/08/2026 14:49:10
Loading file Agents.Message.LLMRequest.cls as udl
Compiling class Agents.Message.LLMRequest
Compiling table Agents_Message.LLMRequest
Compiling routine Agents.Message.LLMRequest.1
Load finished successfully.

Load started on 04/08/2026 14:49:10
Loading file Agents.Message.LLMResponse.cls as udl
Compiling class Agents.Message.LLMResponse
Compiling table Agents_Message.LLMResponse
Compiling routine Agents.Message.LLMResponse.1
Load finished successfully.

Load started on 04/08/2026 14:49:11
Loading file Agents.Message.LLMOutput.cls as udl
Compiling class Agents.Message.LLMOutp

'Here are great summer hiking options around Boston, from easy strolls to longer, scenic loops:\n\n- Blue Hills Reservation (Milton/Canton): Skyline Trail (9–10 mi, strenuous, views), Great Blue Hill and Eliot Tower (1–3 mi, moderate), Ponkapoag Bog boardwalk (easy).\n- Middlesex Fells Reservation (Medford/Winchester/Stoneham): Skyline Trail (7.5–8 mi, rocky, great city views), Reservoir and Rock Circuit loops (varied lengths).\n- Lynn Woods Reservation (Lynn): Extensive network with Stone Tower and Dungeon Rock; mix of easy to moderate loops.\n- Walden Pond State Reservation (Concord): 1.7‑mi pond loop (easy); connecting trails into Walden Woods. Crowds/parking fill early.\n- Minute Man National Historical Park (Concord/Lexington): Battle Road Trail (~5 mi, flat, historic sites, good shade).\n- World’s End (Hingham, Trustees of Reservations): 3–4 mi rolling carriage paths with harbor views; timed entry or parking fees on busy days.\n- Noanet Woodlands (Dover, Trustees): Moderate fores

Agents can be fetched using only their name. Adding any other parameters will be treated as agent creation.

In [13]:
Agent('Molly') == molly

ValueError: too many values to unpack (expected 5)

In [ ]:
class AlexResponse(BaseModel):
    message: str
    reasoning: str

class MollyResponse(BaseModel):
    text: str
    reasoning: str

alex = Agent(name='Alex', 
             description='Test Agent 1', 
             system_prompt=Prompt(name='alex_system', text='You are a helpful agent'),
             model='gpt-5',
             toolkits=[utils_toolkit],
             response_format=AlexResponse)

molly = Agent(name='Molly', 
             description='Test Agent 2', 
             system_prompt=Prompt(name='molly_system', text='You are a helpful agent'),
             model='gpt-5',
             toolkits=[utils_toolkit, iris_toolkit],
             response_format=MollyResponse)


Load started on 04/08/2026 14:31:19
Loading file Agents.Message.AlexResponse.cls as udl
Compiling class Agents.Message.AlexResponse
Compiling table Agents_Message.AlexResponse
Compiling routine Agents.Message.AlexResponse.1
Load finished successfully.

Load started on 04/08/2026 14:31:19
Loading file Agents.Message.ToolRequest.cls as udl
Compiling class Agents.Message.ToolRequest
Compiling table Agents_Message.ToolRequest
Compiling routine Agents.Message.ToolRequest.1
Load finished successfully.

Load started on 04/08/2026 14:31:19
Loading file Agents.Message.ToolResponse.cls as udl
Compiling class Agents.Message.ToolResponse
Compiling table Agents_Message.ToolResponse
Compiling routine Agents.Message.ToolResponse.1
Load finished successfully.

Load started on 04/08/2026 14:31:20
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/08/2026 14:31:20
Loading file Agents.Ope

In [ ]:
Production('AgentSpace', [molly, alex]).start()


Load started on 04/08/2026 14:31:22
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/08/2026 14:31:23
Loading file Agents.Utils.Production.cls as udl
Compiling class Agents.Utils.Production
Compiling routine Agents.Utils.Production.1
Load finished successfully.

Load started on 04/08/2026 14:31:23
Loading file Agents.Message.LLMRequest.cls as udl
Compiling class Agents.Message.LLMRequest
Compiling table Agents_Message.LLMRequest
Compiling routine Agents.Message.LLMRequest.1
Load finished successfully.

Load started on 04/08/2026 14:31:23
Loading file Agents.Message.LLMResponse.cls as udl
Compiling class Agents.Message.LLMResponse
Compiling table Agents_Message.LLMResponse
Compiling routine Agents.Message.LLMResponse.1
Load finished successfully.

Load started on 04/08/2026 14:31:23
Loading file Agents.Message.LLMOutput.cls as udl
Compiling class Agents.Message.LLMOutp

In [ ]:
molly(message='Which tables do we have in IRIS in the Agents namespace?')

'{"text": "The Agents namespace contains these tables: SQLUser.Agent, SQLUser.AgentToolkit, SQLUser.Chat, SQLUser.Prompt, SQLUser.TestModel, SQLUser.Toolkit, SQLUser.ToolUsage, SQLUser.Usage.", "reasoning": "Used the prior IRIS list_tables tool result, which returned the tables for the Agents namespace without error."}'

In [ ]:
molly(message='What is the weather today?', chat=context)

'{"text": "Today\'s weather in Washington DC: Cloudy with a high of 26\\u00b0 and a low of 13\\u00b0.", "reasoning": "Used the provided weather data for Washington DC indicating sky conditions and temperature range."}'

In [ ]:
molly(message='Recommend some good food spots for lunch', chat='travel')

'{"text": "Here are solid DC lunch picks across styles and budgets:\\n- Le Diplomate (Logan Circle): Classic French bistro, great bread basket and steak frites.\\n- Zaytinya (Penn Quarter): Mediterranean mezze by Jos\\u00e9 Andr\\u00e9s; good for sharing and quick service at lunch.\\n- Old Ebbitt Grill (Downtown/near White House): Historic spot; oysters and crab cakes, big menu, easy for groups.\\n- Daikaya Ramen (Chinatown): Sapporo-style ramen; fast, flavorful. Upstairs izakaya is fun too (opens later).\\n- Founding Farmers (Foggy Bottom): Farm-to-table American; wide menu, reliable for midday.\\n- Shouk (Multiple locations): Plant-based Israeli street food; fast-casual bowls and pitas.\\n- Ben\\u2019s Chili Bowl (U Street): DC classic\\u2014half-smoke with chili; quick, iconic.\\n- The Salt Line (Navy Yard): New England-style seafood; great lobster rolls and oysters, waterside seating.\\n- Maketto (H St NE): Cambodian/Thai flavors, coffee and bakery; stylish but relaxed.\\n- Union M

In [ ]:
class Restaurant(BaseModel):
    name: str
    cuisine: str

class TasteAtlas(BaseModel):
    restaurants: list[Restaurant]
    reasoning: str

molly(message='What are some places I would like? I tend to like Italian and Asian cuisines', response_format=TasteAtlas, chat='travel')


Load started on 04/08/2026 14:32:29
Loading file Agents.Message.Restaurant.cls as udl
Compiling class Agents.Message.Restaurant
Compiling routine Agents.Message.Restaurant.1
Load finished successfully.

Load started on 04/08/2026 14:32:29
Loading file Agents.Message.TasteAtlas.cls as udl
Compiling class Agents.Message.TasteAtlas
Compiling table Agents_Message.TasteAtlas
Compiling routine Agents.Message.TasteAtlas.1
Load finished successfully.


'{"restaurants": [{"name": "L\'Ardente", "cuisine": "Italian"}, {"name": "Sfoglina", "cuisine": "Italian"}, {"name": "Osteria Morini", "cuisine": "Italian"}, {"name": "The Red Hen", "cuisine": "Italian"}, {"name": "Centrolina", "cuisine": "Italian"}, {"name": "2 Amys", "cuisine": "Italian"}, {"name": "Daikaya Ramen", "cuisine": "Japanese"}, {"name": "Sushi Taro", "cuisine": "Japanese"}, {"name": "Anju", "cuisine": "Korean"}, {"name": "Thip Khao", "cuisine": "Laotian"}, {"name": "Tiger Fork", "cuisine": "Chinese"}, {"name": "Maketto", "cuisine": "Cambodian/Thai"}], "reasoning": "Since you\'re in Washington DC and prefer Italian and Asian cuisines, here are strong local picks you may enjoy."}'

In [ ]:
Chat('travel').usage()

'{"input_tokens": 6084, "output_tokens": 10218, "output_reasoning_tokens": 8384, "total_tokens": 16302}'

In [ ]:
Production('AgentSpace').usage()

{'input_tokens': 6084,
 'output_tokens': 10218,
 'output_reasoning_tokens': 8384,
 'total_tokens': 16302}

In [ ]:
molly.usage()

{'input_tokens': 6084,
 'output_tokens': 10218,
 'output_reasoning_tokens': 8384,
 'total_tokens': 16302}

In [ ]:
Production('AgentSpace').usage(agents=[Agent('Molly')])

{'input_tokens': 6084,
 'output_tokens': 10218,
 'output_reasoning_tokens': 8384,
 'total_tokens': 16302}

In [ ]:
# Production('AgentSpace').delete()

Deleted production: User.AgentSpace

Deleting class Agents.REST.Dispatch.AgentSpaceCleaned up production-owned artifacts for: AgentSpace


In [ ]:
# Agent('Molly').delete()
# try:
#     molly('Hello')
# except KeyError as e:
#     print(e)


Deleting class Agents.Gateway.MollyService
Deleting class Agents.Process.Molly"No Agent found for 'Molly'"


In [ ]:
# from agents.utils import get_connection

# conn = get_connection()
# cur = conn.cursor()

# cur.execute("DROP TABLE SQLUser.AgentToolkit")
# cur.execute("DROP TABLE SQLUser.Agent")
# cur.execute("DROP TABLE SQLUser.ToolUsage")
# cur.execute("DROP TABLE SQLUser.Usage")
# conn.commit()